# Connect DuckDB to all 12 months

## Import library

In [1]:
import duckdb
from pathlib import Path

In [2]:
PARQUET_DIR = Path("../data/parquet")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(parents = True, exist_ok = True)

con = duckdb.connect()

In [3]:
con.execute(f"""
    CREATE OR REPLACE VIEW rideshare AS

    SELECT *
    FROM read_parquet(
        '{PARQUET_DIR}/rideshare_2025_*.parquet',
        union_by_name = true
    )
""")

### Check how many row in the data set

In [4]:
con.sql("""
    SELECT COUNT(*)
    FROM rideshare
""")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     55200184 │
└──────────────┘

There are over 55 million rows in the dataset

### Check Missing Values

Because this dataset is very large, I applied some filters before downloading it to reduce the file size. I removed rows with missing `trip_id` values and limited pickup locations to trips within Chicago.

The only missing values I kept are in the drop-off location columns. These missing values are acceptable because they represent trips where the drop-off location is outside the Chicago area.

In [17]:
columns = con.sql("""
    DESCRIBE rideshare
""").df()["column_name"].tolist()

In [18]:
missing_query = ",\n".join(
    [
        f'''
        COUNT(*) FILTER (
            WHERE "{col}" IS NULL
               OR TRIM("{col}") = ''
        ) AS "{col}_missing"
        '''
        for col in columns
    ]
)

missing_values = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        {missing_query}
    FROM rideshare
""").df()

missing_values

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,trip_id_missing,trip_start_timestamp_missing,trip_end_timestamp_missing,trip_seconds_missing,trip_miles_missing,pickup_census_tract_missing,pickup_community_area_missing,dropoff_community_area_missing,fare_missing,tip_missing,additional_charges_missing,trip_total_missing,pickup_centroid_location_missing,dropoff_centroid_location_missing
0,55200184,0,0,0,2585,0,0,311422,3604610,121970,121970,121970,121970,0,3347721


There are 121,970 rows with missing `fare` values and 2,582 rows with missing `trip_seconds` values. Since the dataset contains more than 55 million rows, these missing values represent only about 0.22% of the data. Therefore, I decided to remove these rows before continuing the analysis.

### Drop the missing value in fare and trip_second

In [28]:
con.execute("""
    CREATE OR REPLACE VIEW rideshare_dropped AS

    SELECT *
    FROM rideshare
    WHERE fare IS NOT NULL
      AND TRIM(fare) <> ''
      AND trip_seconds IS NOT NULL
      AND TRIM(trip_seconds) <> ''
""")

### Inspect raw values

In [29]:
con.sql("""
    SELECT
        trip_start_timestamp,
        trip_seconds,
        trip_miles,
        pickup_community_area,
        dropoff_community_area,
        fare,
        tip,
        additional_charges,
        trip_total,
    FROM rideshare_dropped
    LIMIT 20
""")

┌────────────────────────┬──────────────┬────────────┬───────────────────────┬────────────────────────┬─────────┬─────────┬────────────────────┬────────────┐
│  trip_start_timestamp  │ trip_seconds │ trip_miles │ pickup_community_area │ dropoff_community_area │  fare   │   tip   │ additional_charges │ trip_total │
│        varchar         │   varchar    │  varchar   │        varchar        │        varchar         │ varchar │ varchar │      varchar       │  varchar   │
├────────────────────────┼──────────────┼────────────┼───────────────────────┼────────────────────────┼─────────┼─────────┼────────────────────┼────────────┤
│ 01/31/2025 11:45:00 PM │ 831          │ 3.1        │ 24                    │ 8                      │ $7.5    │ $0      │ $4.5               │ $12        │
│ 01/31/2025 11:45:00 PM │ 296          │ 0.7        │ 8                     │ 8                      │ $10     │ $0      │ $3.62              │ $13.62     │
│ 01/31/2025 11:45:00 PM │ 831          │ 3.0       

### Check the datatype

In [30]:
con.sql("""
    DESCRIBE
    SELECT *
    FROM rideshare_dropped
""")

┌───────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name        │ column_type │  null   │   key   │ default │  extra  │
│          varchar          │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ trip_id                   │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ trip_start_timestamp      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ trip_end_timestamp        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ trip_seconds              │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ trip_miles                │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ pickup_census_tract       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ pickup_community_area     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ dropoff_community_area    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NU

### Convert the datatype

| Column | Type |
|---|---|
| `trip_id` | `VARCHAR` |
| `trip_start_timestamp` | `TIMESTAMP` |
| `trip_end_timestamp` | `TIMESTAMP` |
| `trip_seconds` | `BIGINT` |
| `trip_miles` | `DOUBLE` |
| `pickup_census_tract` | `VARCHAR` |
| `pickup_community_area` | `INTEGER` |
| `dropoff_community_area` | `INTEGER` |
| `fare` | `DECIMAL(12,2)` |
| `tip` | `DECIMAL(12,2)` |
| `additional_charges` | `DECIMAL(12,2)` |
| `trip_total` | `DECIMAL(12,2)` |
| centroid locations | `VARCHAR` |

In [31]:
con.execute("""
    CREATE OR REPLACE VIEW rideshare_typed AS

    SELECT
        trip_id,

        COALESCE(
            TRY_CAST(trip_start_timestamp AS TIMESTAMP),
            TRY_STRPTIME(
                trip_start_timestamp,
                '%m/%d/%Y %I:%M:%S %p')) AS trip_start_timestamp,

        COALESCE(
            TRY_CAST(trip_end_timestamp AS TIMESTAMP),
            TRY_STRPTIME(
                trip_end_timestamp,
                '%m/%d/%Y %I:%M:%S %p')) AS trip_end_timestamp,

        TRY_CAST(
            REPLACE(trip_seconds, ',', '')
            AS BIGINT) AS trip_seconds,

        TRY_CAST(
            REPLACE(trip_miles, ',', '')
            AS DOUBLE) AS trip_miles,

        pickup_census_tract,

        TRY_CAST(
            pickup_community_area
            AS INTEGER) AS pickup_community_area,

        TRY_CAST(
            dropoff_community_area
            AS INTEGER) AS dropoff_community_area,

        TRY_CAST(
            REPLACE(
                REPLACE(fare, '$', ''),
                ',', ''
            )
            AS DECIMAL(12, 2)) AS fare,

        TRY_CAST(
            REPLACE(
                REPLACE(tip, '$', ''),
                ',', ''
            )
            AS DECIMAL(12, 2)) AS tip,

        TRY_CAST(
            REPLACE(
                REPLACE(additional_charges, '$', ''),
                ',', ''
            )
            AS DECIMAL(12, 2)) AS additional_charges,

        TRY_CAST(
            REPLACE(
                REPLACE(trip_total, '$', ''),
                ',', ''
            )
            AS DECIMAL(12, 2)) AS trip_total,

        pickup_centroid_location,
        dropoff_centroid_location

    FROM rideshare_dropped
""")

In [33]:
con.sql("""
    DESCRIBE
    SELECT *
    FROM rideshare_typed
""")

┌───────────────────────────┬───────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name        │  column_type  │  null   │   key   │ default │  extra  │
│          varchar          │    varchar    │ varchar │ varchar │ varchar │ varchar │
├───────────────────────────┼───────────────┼─────────┼─────────┼─────────┼─────────┤
│ trip_id                   │ VARCHAR       │ YES     │ NULL    │ NULL    │ NULL    │
│ trip_start_timestamp      │ TIMESTAMP     │ YES     │ NULL    │ NULL    │ NULL    │
│ trip_end_timestamp        │ TIMESTAMP     │ YES     │ NULL    │ NULL    │ NULL    │
│ trip_seconds              │ BIGINT        │ YES     │ NULL    │ NULL    │ NULL    │
│ trip_miles                │ DOUBLE        │ YES     │ NULL    │ NULL    │ NULL    │
│ pickup_census_tract       │ VARCHAR       │ YES     │ NULL    │ NULL    │ NULL    │
│ pickup_community_area     │ INTEGER       │ YES     │ NULL    │ NULL    │ NULL    │
│ dropoff_community_area    │ INTEGER       │ YES     

In [34]:
con.sql("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE trip_start_timestamp IS NULL
        ) AS missing_start_time,

        COUNT(*) FILTER (
            WHERE trip_seconds IS NULL
        ) AS missing_seconds,

        COUNT(*) FILTER (
            WHERE trip_miles IS NULL
        ) AS missing_miles,

        COUNT(*) FILTER (
            WHERE pickup_community_area IS NULL
        ) AS missing_pickup_area,

        COUNT(*) FILTER (
            WHERE dropoff_community_area IS NULL
        ) AS missing_dropoff_area,

        COUNT(*) FILTER (
            WHERE fare IS NULL
        ) AS missing_fare,

        COUNT(*) FILTER (
            WHERE trip_total IS NULL
        ) AS missing_trip_total

    FROM rideshare_typed
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┬────────────────────┬─────────────────┬───────────────┬─────────────────────┬──────────────────────┬──────────────┬────────────────────┐
│ total_rows │ missing_start_time │ missing_seconds │ missing_miles │ missing_pickup_area │ missing_dropoff_area │ missing_fare │ missing_trip_total │
│   int64    │       int64        │      int64      │     int64     │        int64        │        int64         │    int64     │       int64        │
├────────────┼────────────────────┼─────────────────┼───────────────┼─────────────────────┼──────────────────────┼──────────────┼────────────────────┤
│   55075632 │                  0 │               0 │             0 │              304839 │              3566844 │            0 │                  0 │
└────────────┴────────────────────┴─────────────────┴───────────────┴─────────────────────┴──────────────────────┴──────────────┴────────────────────┘



## Impute missing value

### Impute Missing Value in Pickup Area

The dataset was filtered so that all pickup locations are within Chicago, and there are no missing values in `pickup_centroid_location`. However, there are still 304,839 missing values in `pickup_community_area`.

Because the pickup centroid location is available, I will use the geographic coordinates to identify the corresponding Chicago community area and fill in the missing `pickup_community_area` values.

#### Load Chicago Community Area boundaries

In [36]:
REFERENCE_DIR = Path("../data/reference")

REFERENCE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

#### Load DuckDB's spatial extension

In [38]:
con.execute("""
    INSTALL spatial;
    LOAD spatial;
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

#### Load the community-area polygons

In [39]:
community_file = (REFERENCE_DIR / "community_areas.geojson")

con.execute(f"""
    CREATE OR REPLACE TABLE community_areas AS

    SELECT *
    FROM ST_Read(
        '{community_file}'
    )
""")

#### Check the datatype and columns

In [41]:
con.sql("""
    DESCRIBE community_areas
""").show()

┌─────────────┬──────────────────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │       column_type        │  null   │   key   │ default │  extra  │
│   varchar   │         varchar          │ varchar │ varchar │ varchar │ varchar │
├─────────────┼──────────────────────────┼─────────┼─────────┼─────────┼─────────┤
│ OGC_FID     │ BIGINT                   │ YES     │ NULL    │ NULL    │ NULL    │
│ :id         │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ :version    │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ :created_at │ TIMESTAMP WITH TIME ZONE │ YES     │ NULL    │ NULL    │ NULL    │
│ :updated_at │ TIMESTAMP WITH TIME ZONE │ YES     │ NULL    │ NULL    │ NULL    │
│ area_numbe  │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ community   │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ area_num_1  │ VARCHAR                  │ YES     │ NULL    │ NULL    │ NULL    │
│ sh

In [43]:
con.sql("""
    SELECT
        area_numbe,
        community
    FROM community_areas
    LIMIT 10
""").show()

┌────────────┬─────────────────┐
│ area_numbe │    community    │
│  varchar   │     varchar     │
├────────────┼─────────────────┤
│ 1          │ ROGERS PARK     │
│ 2          │ WEST RIDGE      │
│ 3          │ UPTOWN          │
│ 4          │ LINCOLN SQUARE  │
│ 5          │ NORTH CENTER    │
│ 6          │ LAKE VIEW       │
│ 7          │ LINCOLN PARK    │
│ 8          │ NEAR NORTH SIDE │
│ 9          │ EDISON PARK     │
│ 10         │ NORWOOD PARK    │
└────────────┴─────────────────┘
  10 rows            2 columns



#### Create pickup location lookup

Instead of join all 55 million rows. I get only unique centroid locations where the pickup community area is missing

In [50]:
con.execute("""
    CREATE OR REPLACE TABLE missing_pickup_locations AS
    SELECT DISTINCT pickup_centroid_location
    FROM rideshare_typed
    WHERE pickup_community_area is NULL
    AND pickup_centroid_location IS NOT NULL
""")

Check how many unique locations need to be matched

In [51]:
con.sql("""
    SELECT COUNT(*) AS unique_locations
    FROM missing_pickup_locations
""").show()

┌──────────────────┐
│ unique_locations │
│      int64       │
├──────────────────┤
│                1 │
└──────────────────┘



There is only one unique location with a missing `pickup_community_area`, so let's find which community area contains that centroid.

In [52]:
con.sql("""
    SELECT
        pickup_centroid_location,
        COUNT(*) AS trip_count
    FROM rideshare_typed
    WHERE pickup_community_area IS NULL
    GROUP BY pickup_centroid_location
""").show()

┌──────────────────────────────────────┬────────────┐
│       pickup_centroid_location       │ trip_count │
│               varchar                │   int64    │
├──────────────────────────────────────┼────────────┤
│ POINT (-87.8773053996 41.9827750091) │     304839 │
└──────────────────────────────────────┴────────────┘



There are 304,839 missing `pickup_community_area` values, and all of them share the same centroid location, approximately `(-87.87, 41.98)`. Since this count matches the total number of missing pickup community area values, we only need to identify the community area that contains this centroid.

In [55]:
con.sql("""
    SELECT
        area_numbe,
        community
    FROM community_areas
    WHERE ST_Within(
        ST_GeomFromText(
            'POINT (-87.8773053996 41.9827750091)'
        ),
        geom
    )
""").show()

┌────────────┬───────────┐
│ area_numbe │ community │
│  varchar   │  varchar  │
├────────────┼───────────┤
│ 76         │ OHARE     │
└────────────┴───────────┘



#### Fill missing pickup community area

The centroid `(-87.8773, 41.9828)` is located in Community Area 76, O'Hare. Since all 304,839 missing `pickup_community_area` values share this same centroid, I filled these missing values with `76`.

In [57]:
con.execute("""
    CREATE OR REPLACE VIEW rideshare_pickup_imputed AS

    SELECT
        * EXCLUDE (pickup_community_area),

        COALESCE(
            pickup_community_area,
            76
        ) AS pickup_community_area

    FROM rideshare_typed
""")

### Fill the missing value in drop off area

The missing dropoff_community_area values represent trips where the drop-off location is outside Chicago. Therefore, I replaced these missing values with -1 to distinguish them from Chicago community areas 1–77.

In [59]:
con.execute("""
    CREATE OR REPLACE VIEW rideshare_imputed AS

    SELECT
        * EXCLUDE (dropoff_community_area),
        COALESCE (dropoff_community_area, -1) AS dropoff_community_area
    FROM rideshare_pickup_imputed
""")

### Check the dataset

In [63]:
columns = con.sql("""
    DESCRIBE rideshare_imputed
""").df()["column_name"].tolist()

missing_query = ",\n".join(
    [f'COUNT(*) FILTER (WHERE "{col}" IS NULL) AS "{col}"'
        for col in columns
    ])

missing_values = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        {missing_query}
    FROM rideshare_imputed
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [64]:
missing_df = (
    missing_values
    .drop(columns="total_rows")
    .T
    .reset_index()
)

missing_df.columns = [
    "column",
    "missing_count"
]

missing_df

,column,missing_count
0,trip_id,0
1,trip_start_timestamp,0
2,trip_end_timestamp,0
3,trip_seconds,0
4,trip_miles,0
5,pickup_census_tract,0
6,fare,0
7,tip,0
8,additional_charges,0
9,trip_total,0


There are no more missing values in `trip_seconds`, `fare`, `pickup_community_area`, or `dropoff_community_area`. The only remaining missing values are in `dropoff_centroid_location`, but this column is not necessary for our analysis.